In [ ]:
from deepagents import create_deep_agent

In [ ]:
# Setup 1/4: config + secrets
import os
import asyncio
import base64
import json
import re
from dotenv import load_dotenv

load_dotenv(override=True)

MODEL = os.getenv("MODEL", "anthropic:claude-sonnet-5")
BASE = "https://parabank.parasoft.com/parabank"
ALLOWED_HOSTS = {"parabank.parasoft.com"}
SECRETS = {"username": "PARABANK_USERNAME", "password": "PARABANK_PASSWORD"}


def resolve_secret(name: str) -> str:
    """Look up a secret by name. Raises on unknown name or empty value."""
    if name not in SECRETS:
        raise KeyError(f"unknown secret name: {name!r}")
    value = os.environ.get(SECRETS[name], "")
    if not value:
        raise RuntimeError(f"env var {SECRETS[name]} is empty or not set")
    return value


print("model:", MODEL, "| base:", BASE)

In [ ]:
# Setup 2/4: browser
# Open a visible browser on the login page (skips if one is already open in this kernel).
from playwright.async_api import async_playwright

if "page" not in globals():
    pw = await async_playwright().start()
    browser = await pw.chromium.launch(headless=False)
    context = await browser.new_context(viewport={"width": 1280, "height": 900})
    page = await context.new_page()
await page.goto(f"{BASE}/index.htm")
print("opened:", page.url)


In [ ]:
# Setup 3/4: domain guard
from urllib.parse import urlparse


def host_allowed(url: str) -> bool:
    if url == "about:blank":
        return True
    return urlparse(url).hostname in ALLOWED_HOSTS

In [ ]:
# Setup 4/4: numbered screenshot (PlaywrightSurface)
from dataclasses import dataclass

OBSERVE_JS = """
() => {
  document.querySelectorAll('[data-cua-ref]').forEach(e => e.removeAttribute('data-cua-ref'));
  const sel = 'a[href], button, input:not([type=hidden]), select, textarea, [role=button], [role=link], [onclick]';
  const roleOf = (el) => {
    const r = el.getAttribute('role'); if (r) return r;
    const t = el.tagName.toLowerCase();
    if (t === 'a') return 'link';
    if (t === 'button') return 'button';
    if (t === 'select') return 'combobox';
    if (t === 'textarea') return 'textbox';
    if (t === 'input') {
      const ty = (el.getAttribute('type') || 'text').toLowerCase();
      if (['submit', 'button', 'reset', 'image'].includes(ty)) return 'button';
      if (ty === 'checkbox') return 'checkbox';
      if (ty === 'radio') return 'radio';
      return 'textbox';
    }
    return 'generic';
  };
  const nameOf = (el) => {
    const aria = el.getAttribute('aria-label'); if (aria) return aria.trim();
    if (el.labels && el.labels.length) return el.labels[0].innerText.trim();
    const t = el.tagName.toLowerCase();
    const ty = (el.getAttribute('type') || '').toLowerCase();
    if (t === 'input' && ['submit', 'button', 'reset'].includes(ty)) return (el.value || '').trim();
    const txt = (el.innerText || '').trim(); if (txt) return txt.slice(0, 80);
    return (el.getAttribute('placeholder') || el.getAttribute('title') ||
            el.getAttribute('alt') || el.getAttribute('name') || '').trim();
  };
  const items = [];
  document.querySelectorAll(sel).forEach((el) => {
    const r = el.getBoundingClientRect();
    const st = getComputedStyle(el);
    if (r.width < 2 || r.height < 2 || st.visibility === 'hidden' || st.display === 'none') return;
    const ref = items.length + 1;
    el.setAttribute('data-cua-ref', String(ref));
    items.push({
      ref, role: roleOf(el), name: nameOf(el),
      x: r.x, y: r.y, w: r.width, h: r.height,
      inViewport: r.bottom > 0 && r.top < innerHeight && r.right > 0 && r.left < innerWidth,
    });
  });
  return items;
}
"""

DRAW_JS = """
(items) => {
  const box = document.createElement('div');
  box.id = '__cua_overlay';
  box.style.cssText = 'position:fixed;inset:0;pointer-events:none;z-index:2147483647';
  items.filter(i => i.inViewport).forEach(i => {
    const b = document.createElement('div');
    b.style.cssText = `position:fixed;left:${i.x}px;top:${i.y}px;width:${i.w}px;height:${i.h}px;border:2px solid red;box-sizing:border-box`;
    const l = document.createElement('span');
    l.textContent = i.ref;
    l.style.cssText = 'position:absolute;left:0;top:-14px;background:red;color:#fff;font:bold 11px monospace;padding:0 3px';
    b.appendChild(l);
    box.appendChild(b);
  });
  document.body.appendChild(box);
}
"""

CLEAR_JS = "() => { const o = document.getElementById('__cua_overlay'); if (o) o.remove(); }"


def format_elements(elements: list[dict]) -> str:
    lines = []
    for e in elements:
        line = f'[{e["ref"]}] {e["role"]} "{e["name"]}"'
        if not e["inViewport"]:
            line += " (below the fold)"
        lines.append(line)
    return "\n".join(lines)


@dataclass
class Observation:
    png: bytes
    elements: list[dict]
    url: str
    title: str

    def as_text(self) -> str:
        return f"URL: {self.url}\nTitle: {self.title}\nElements:\n{format_elements(self.elements)}"


class PlaywrightSurface:
    """The only place that touches Playwright. Agent tools talk to this."""

    def __init__(self, page):
        self.page = page
        self.last_elements: list[dict] = []

    async def observe(self) -> Observation:
        elements = await self.page.evaluate(OBSERVE_JS)
        await self.page.evaluate(DRAW_JS, elements)
        png = await self.page.screenshot()
        await self.page.evaluate(CLEAR_JS)
        self.last_elements = elements
        return Observation(png, elements, self.page.url, await self.page.title())

    def name_of(self, ref: int) -> str | None:
        for e in self.last_elements:
            if e["ref"] == ref:
                return e["name"]
        return None

In [ ]:
# STEP 1: scanner patch (dropdown options + submit flag). Run right after the surface cell.
OBSERVE_JS = OBSERVE_JS.replace(
    "inViewport: r.bottom",
    "options: el.tagName === 'SELECT' ? Array.from(el.options).map(o => o.text.trim()).slice(0, 15) : null,\n"
    "      submit: (el.tagName === 'BUTTON' && el.type !== 'button') || (el.tagName === 'INPUT' && ['submit', 'image'].includes(el.type)),\n"
    "      inViewport: r.bottom",
)
assert "options:" in OBSERVE_JS and "submit:" in OBSERVE_JS, "patch did not apply"


def format_elements(elements: list[dict]) -> str:
    lines = []
    for e in elements:
        line = f'[{e["ref"]}] {e["role"]} "{e["name"]}"'
        if e.get("options"):
            line += f' options={e["options"]}'
        if not e["inViewport"]:
            line += " (below the fold)"
        lines.append(line)
    return "\n".join(lines)


In [ ]:
# STEP 2: banner, takeover, Approve / Reject / Take over buttons, hand_off
import asyncio
from langgraph.types import Command

HANDBACK = {"event": None}

DECISION_JS = """(info) => new Promise(resolve => {
  const bar = document.createElement('div');
  bar.style.cssText = 'position:fixed;top:0;left:0;right:0;z-index:2147483647;background:#111;color:#fff;font:14px sans-serif;padding:10px;display:flex;flex-direction:column;gap:8px;align-items:center';
  const t = document.createElement('b'); t.textContent = info.title;
  const d = document.createElement('div'); d.textContent = info.details;
  const row = document.createElement('div'); row.style.cssText = 'display:flex;gap:12px';
  [['Approve', 'a'], ['Reject', 'r'], ['Take over', 't']].forEach(([label, val]) => {
    const b = document.createElement('button'); b.textContent = label;
    b.style.cssText = 'padding:6px 14px;font-size:14px;cursor:pointer';
    b.onclick = () => { bar.remove(); resolve(val); };
    row.appendChild(b);
  });
  bar.append(t, d, row);
  document.body.appendChild(bar);
})"""

BANNER_JS = """
(() => {
  const render = () => {
    if (sessionStorage.getItem('cua_takeover') !== '1' || document.getElementById('__cua_banner')) return;
    const q = sessionStorage.getItem('cua_question') || '';
    const b = document.createElement('div');
    b.id = '__cua_banner';
    b.style.cssText = 'position:fixed;top:0;left:0;right:0;z-index:2147483647;background:#c00;color:#fff;font:14px sans-serif;padding:8px;display:flex;gap:12px;align-items:center;justify-content:center';
    const label = document.createElement('b');
    label.textContent = 'YOU are in control.';
    const msg = document.createElement('span');
    msg.textContent = q ? 'Agent asks: ' + q : 'Do the step in this page.';
    const btn = document.createElement('button');
    btn.textContent = 'Done, hand back to agent';
    btn.style.cssText = 'padding:6px 14px;font-size:14px;cursor:pointer';
    btn.onclick = () => window.__cua_handback();
    b.append(label, msg, btn);
    document.documentElement.appendChild(b);
  };
  if (document.readyState === 'loading') document.addEventListener('DOMContentLoaded', render); else render();
})();
"""

if "_takeover_ready" not in globals():
    async def _handback():
        if HANDBACK["event"]:
            HANDBACK["event"].set()
    await page.expose_function("__cua_handback", _handback)
    await page.add_init_script(BANNER_JS)
    _takeover_ready = True


async def human_takeover(question: str = "") -> str:
    """Give the live browser to a human. Returns when they click 'Done' in the page."""
    HANDBACK["event"] = asyncio.Event()
    visited = []
    on_nav = lambda frame: visited.append(frame.url) if frame == page.main_frame else None
    page.on("framenavigated", on_nav)
    await page.evaluate("([q]) => { sessionStorage.setItem('cua_takeover', '1'); sessionStorage.setItem('cua_question', q); }", [question])
    await page.evaluate(BANNER_JS)
    await HANDBACK["event"].wait()
    page.remove_listener("framenavigated", on_nav)
    await page.evaluate("sessionStorage.removeItem('cua_takeover'); sessionStorage.removeItem('cua_question'); document.getElementById('__cua_banner')?.remove()")
    text = (await page.inner_text("body"))[:300].replace("\n", " ")
    return f"Pages the human visited: {visited or 'none'}. Page now: {page.url}. Page text: {text}"


def approval_info(args: dict) -> dict:
    name = surface.name_of(args["ref"])
    fields = "; ".join(f"{k}: {v}" for k, v in TYPED.items()) or "(nothing typed)"
    return {"title": f"Agent wants to click '{name}' on {page.url.split('/')[-1]}", "details": f"Values it entered: {fields}"}


async def hand_off(out, cfg):
    while out.interrupts:
        info = approval_info(out.interrupts[0].value["action_requests"][0]["args"])
        print("\n=== HUMAN NEEDED ===", info["title"], "|", info["details"])
        choice = await page.evaluate(DECISION_JS, info)
        if choice == "a":
            decision = {"type": "approve"}
        elif choice == "t":
            report = await human_takeover()
            decision = {"type": "reject", "message": f"A human completed this step manually in the browser. {report} Do NOT click again. Check the result from the page text, then call finish."}
        else:
            decision = {"type": "reject", "message": "DECLINED by a human. Do not retry or work around it. Call finish with 'DECLINED:' and stop."}
        out = await agent_hitl.ainvoke(Command(resume={"decisions": [decision]}), config=cfg, version="v2")
    return out


In [ ]:
# STEP 3: browser tools
import asyncio, base64, functools
from langchain.tools import tool

ACT_LOCK = asyncio.Lock()


def one_at_a_time(fn):
    @functools.wraps(fn)
    async def wrapper(*a, **k):
        async with ACT_LOCK:
            return await fn(*a, **k)
    return wrapper


TYPED: dict[str, str] = {}      # what the agent entered, by field name (shown in the approval bar)
GIVEN = {"text": ""}            # what the user actually gave us; set before each run
RESULT: dict = {}
DENY_LINKS = ("register", "lookup", "admin")
SENSITIVE_WORDS = ("ssn", "password", "social")


async def _click(self, ref: int):
    await self.page.locator(f'[data-cua-ref="{ref}"]').click(timeout=5000)
    await self.page.wait_for_timeout(600)
    try:
        await self.page.wait_for_load_state("load", timeout=5000)
    except Exception:
        pass

async def _type_text(self, ref: int, text: str):
    await self.page.locator(f'[data-cua-ref="{ref}"]').fill(text, timeout=5000)

async def _page_text(self) -> str:
    return (await self.page.inner_text("body"))[:4000]

PlaywrightSurface.click = _click
PlaywrightSurface.type_text = _type_text
PlaywrightSurface.page_text = _page_text
surface = PlaywrightSurface(page)


def _blocks(prefix: str, obs) -> list[dict]:
    """Package text and a screenshot as one tool result the model can read."""
    return [
        {"type": "text", "text": f"{prefix}\n{obs.as_text()}"},
        {"type": "image", "base64": base64.b64encode(obs.png).decode(), "mime_type": "image/png"},
    ]


async def _ask_for_value(field: str, kind: str) -> list:
    """The agent tried to enter a value the user never gave. Hand the browser to a human."""
    report = await human_takeover(
        f"I need a value for '{field}' and you did not give me one. Please {kind} it yourself in the page, then click Done."
    )
    return _blocks(
        f"A human entered the value for '{field}' themselves. Do NOT type it again. {report}",
        await surface.observe(),
    )


@tool(parse_docstring=True)
@one_at_a_time
async def observe() -> list:
    """Look at the current page.

    Returns:
        A screenshot with red numbered boxes, plus a text list of the numbered elements.
    """
    return _blocks("Current page.", await surface.observe())


SAFE_SUBMITS = {"log in", "find transactions"}   # buttons that do not change data
AUTO_LIMIT = None                                # None = always ask a human. Later: 500.0 for small transfers
DECLINED: set[str] = set()                       # buttons a human refused; never clicked again this run


def _amount() -> float:
    raw = TYPED.get("amount", "").replace("$", "").replace(",", "")
    try:
        return float(raw)
    except ValueError:
        return float("inf")                      # unknown amount counts as risky


def needs_human(ref: int) -> bool:
    """Every button except a short safe list needs a human. Links (navigation) run freely."""
    el = next((e for e in surface.last_elements if e["ref"] == ref), None)
    role = (el or {}).get("role")
    name = ((el or {}).get("name") or "").strip().lower()
    risky = bool(el) and (bool(el.get("submit")) or role == "button") and name not in SAFE_SUBMITS
    if risky and "transfer" in name and AUTO_LIMIT is not None and _amount() <= AUTO_LIMIT:
        risky = False
    print(f"approval check -> role={role!r} name={name!r} risky={risky}")
    return risky


@tool(parse_docstring=True)
@one_at_a_time
async def click(ref: int) -> list:
    """Click an element on the page.

    Buttons that change data ask a human for approval first. You do not need to do anything for that.

    Args:
        ref: Number of the element in the latest screenshot and list.

    Returns:
        The new page state. If a human declined, the result says DECLINED and you must stop.
    """
    name = (surface.name_of(ref) or "").strip().lower()
    if any(w in name for w in DENY_LINKS):
        return _blocks(f"DENIED: '{name}' is not allowed.", await surface.observe())
    if name in DECLINED:
        return _blocks("DECLINED earlier by a human. Do not retry. Call finish with 'DECLINED:' and stop.", await surface.observe())
    if needs_human(ref):
        choice = await page.evaluate(DECISION_JS, approval_info({"ref": ref}))   # buttons appear in the browser
        if choice == "r":
            DECLINED.add(name)
            return _blocks("DECLINED by a human. Do not retry or work around it. Call finish with 'DECLINED:' and stop.", await surface.observe())
        if choice == "t":
            report = await human_takeover()
            return _blocks(f"A human completed this step manually in the browser. {report} Do NOT click again. Check the result from the page text, then call finish.", await surface.observe())
        # choice == "a": approved, fall through and click
    try:
        await surface.click(ref)
    except Exception as exc:
        return _blocks(f"CLICK FAILED for [{ref}]: {type(exc).__name__}", await surface.observe())
    if not host_allowed(page.url):
        await page.go_back()
        return _blocks("BLOCKED: left the allowed site. Went back.", await surface.observe())
    return _blocks(f"Clicked [{ref}].", await surface.observe())


@tool(parse_docstring=True)
@one_at_a_time
async def type_text(ref: int, text: str) -> list:
    """Type normal text into an input box.

    Only type values the user gave you in the goal. Never invent a value.

    Args:
        ref: Number of the input box in the latest screenshot and list.
        text: The text to type. It must come from the user's goal.

    Returns:
        The new page state.
    """
    field = (surface.name_of(ref) or f"field {ref}").strip()
    if any(w in field.lower() for w in SENSITIVE_WORDS) or text.strip().lower() not in GIVEN["text"].lower():
        return await _ask_for_value(field, "type")
    try:
        await surface.type_text(ref, text)
    except Exception as exc:
        return _blocks(f"TYPE FAILED for [{ref}]: {type(exc).__name__}", await surface.observe())
    TYPED[field.lower()] = text
    return _blocks(f"Typed into [{ref}].", await surface.observe())


@tool(parse_docstring=True)
@one_at_a_time
async def type_secret(ref: int, name: str) -> list:
    """Type a stored secret into an input box, without ever seeing its value.

    Use this to log in. You give only the secret's name.

    Args:
        ref: Number of the input box in the latest screenshot and list.
        name: Secret name. Either 'username' or 'password'.

    Returns:
        The new page state. The value is never included.
    """
    if name not in SECRETS:
        return _blocks(f"UNKNOWN SECRET '{name}'. Use one of: {list(SECRETS)}", await surface.observe())
    if not host_allowed(page.url):
        return _blocks("REFUSED: this site is not on the allowlist.", await surface.observe())
    try:
        await page.locator(f'[data-cua-ref="{ref}"]').fill(resolve_secret(name), timeout=5000)
    except Exception as exc:
        return _blocks(f"FAILED for [{ref}]: {type(exc).__name__}", await surface.observe())
    return _blocks(f"Typed secret '{name}' into [{ref}].", await surface.observe())


@tool(parse_docstring=True)
@one_at_a_time
async def select_option(ref: int, option: str) -> list:
    """Choose an option in a dropdown.

    Only choose options the user named in the goal. Never guess.

    Args:
        ref: Number of the dropdown in the latest screenshot and list.
        option: Visible text of the option, exactly as shown in its options list.

    Returns:
        The new page state.
    """
    field = (surface.name_of(ref) or f"dropdown {ref}").strip()
    if option.strip().lower() not in GIVEN["text"].lower():
        return await _ask_for_value(field, "choose")
    try:
        await page.locator(f'[data-cua-ref="{ref}"]').select_option(label=option, timeout=5000)
    except Exception as exc:
        return _blocks(f"SELECT FAILED for [{ref}]: {type(exc).__name__}", await surface.observe())
    TYPED[field.lower()] = option
    return _blocks(f"Selected '{option}' in [{ref}].", await surface.observe())


@tool(parse_docstring=True)
@one_at_a_time
async def page_text() -> str:
    """Read the visible text of the whole page (balances, tables, messages).

    Returns:
        The page text, up to 4000 characters.
    """
    return await surface.page_text()


@tool(parse_docstring=True)
@one_at_a_time
async def ask_human(question: str) -> list:
    """Ask a human for help by handing over the browser.

    Use this when you are unsure which element to pick, the page looks unexpected,
    a step failed twice, or you would have to guess. Never guess account choices or amounts.

    Args:
        question: What you are unsure about and what you need the human to do.

    Returns:
        What the human did, and the new page state.
    """
    report = await human_takeover(question)
    return _blocks(f"A human took over and handed back. {report}", await surface.observe())


@tool(parse_docstring=True)
async def finish(summary: str, values: dict[str, str]) -> str:
    """Report the final result and stop.

    If you cannot make progress, start the summary with 'STUCK:'. If a human declined, start it with 'DECLINED:'.

    Args:
        summary: One-line summary of what you did or why you stopped.
        values: The requested facts, for example {"account_id": "13344", "balance": "$100.00"}.

    Returns:
        A confirmation. Stop after this.
    """
    RESULT.clear()
    RESULT.update({"summary": summary, "values": values})
    return "Recorded. Stop now."


BROWSER_TOOLS = [observe, click, type_text, type_secret, select_option, page_text, ask_human, finish]
print("tools ready:", [t.name for t in BROWSER_TOOLS])


In [ ]:
# STEP 4: system prompt and agents (use agent_hitl for anything that changes data)
from langgraph.checkpoint.memory import MemorySaver

SYSTEM_PROMPT = """You are an expert browser operator. You drive a real browser on a banking demo site, and you can search the web for outside facts.

## Browser tools
Every tool result shows a screenshot with red numbered boxes plus a list of numbered elements (dropdowns list their options). Refer to elements only by number. Numbers change after every action, so use the latest list.
- observe: look at the page.
- click(ref), type_text(ref, text), select_option(ref, option): act on elements.
- type_secret(ref, name): type a stored secret ('username' or 'password'). You never see the value.
- page_text: read the visible page text (balances, messages).
- ask_human(question): hand the browser to a human when you are unsure.
- finish(summary, values): report the result, then stop.

## How to work
1. Call observe first. If you see a login form, log in with type_secret, then confirm the account overview appears.
2. Do the task by the shortest path. Read values with page_text and copy them exactly. Never invent a value.
3. Use ONLY values the user gave you in the goal. If a value you need (payee, amount, account, address) is missing, call ask_human. Never make one up.
4. If a tool says a human entered a value, do not type it again. Continue with the next step.
5. Buttons that change data (Send Payment, Transfer, Open New Account, and so on) need a human. Click the button when the form is ready; the system asks the human for approval by itself. If the result says DECLINED, never retry or work around it: call finish with 'DECLINED:'.
6. Make ONE tool call at a time. Do not click into a field before typing.
7. Stay on the banking site. If you are lost or repeat the same action 3 times, call finish with 'STUCK:' and say why.
8. Do not use ls, read_file, write_file, edit_file, delete, glob, grep or task.
"""

internet_search = {"type": "web_search_20260209", "name": "web_search"}

agent = create_deep_agent(
    model=MODEL,
    tools=[internet_search, *BROWSER_TOOLS],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=MemorySaver(),
)

agent_hitl = agent      # same agent: the approval now lives inside the click tool
print("agent ready | approval is enforced inside the click tool")


In [ ]:
# Run a bill payment. Every value below is given by the user, so the agent may type it.
import uuid

GOAL = ("Pay a bill. ")   # <-- change 14232 to YOUR account id
GIVEN["text"] = GOAL
TYPED.clear()
await page.goto(f"{BASE}/overview.htm")
cfg = {"configurable": {"thread_id": f"bill-{uuid.uuid4().hex[:6]}"}, "recursion_limit": 100}
out = await agent.ainvoke({"messages": [{"role": "user", "content": GOAL}]}, config=cfg)
print(out["messages"][-1].content)
